# PyPi Ecosystem Dependency Graph

In [ ]:
import math
import polars as pl
import networkx as nx
import pygraphviz as pgv
import altair as alt
from pyvis.network import Network
from settings import load_settings


In [ ]:
SETTINGS = load_settings()

## 1. Loading the Data

In [ ]:
edges_df = pl.scan_parquet(SETTINGS.dependency_edges_path)

In [ ]:
edges_df.head().collect()

## 2. Aggregate
```
package -> dependency
           versions_count = number of distinct package versions that use that dependency
```

In [ ]:
agg = (
    edges_df
    .group_by(["package_name", "dep_name"])
    .agg([
        pl.col("package_version").n_unique().alias("versions_count"),
        #pl.col("dep_version").n_unique().alias("dep_versions_count"),
    ])
    .sort("versions_count", descending=True)
    .collect()
)

## 3. Build the threshold curve: for each K, count rows left after filtering

Explanation of K (versions_count threshold)

In the dependency graph, edges represent relationships between a package and its dependencies aggregated across multiple package versions.
For each pair `(package_name, dep_name)` we compute a value called `versions_count`, defined as the number of distinct package versions that depend on that dependency.

For example:

taipy-gui ───(3)──▶ pandas

means that three different versions of taipy-gui depend on pandas.

A larger value of versions_count indicates that the dependency appears consistently across multiple releases of the package, suggesting a stronger or more stable relationship.

To reduce noise and improve readability of the graph, we introduce a filtering parameter K.

### Definition of K

K is the minimum number of package versions in which a dependency must appear in order for the edge to be included in the graph.

Formally, we keep only edges where `versions_count ≥ K`

This means:
- K = 1 → keep all dependencies
- K = 2 → keep only dependencies used in at least two versions
- K = 3 → keep only dependencies used in at least three versions
- larger K → keep only very stable / frequent dependencies

As K increases, the number of edges in the graph decreases, but the remaining edges represent more persistent dependency relationships.

We choose K by plotting how the number of edges remaining in the dataset changes as K increases, and selecting a value where the graph becomes readable without removing too much information.

This allows us to balance:
- completeness (small K)
- clarity and stability (large K)

when visualizing the dependency network.

In [ ]:
max_k = int(agg["versions_count"].max())
curve = (
    pl.DataFrame({"k": list(range(1, max_k + 1))})
    .with_columns(
        pl.col("k").map_elements(lambda k: agg.filter(pl.col("versions_count") >= k).height, return_dtype=pl.Int64).alias("rows_remaining")
    )
)
total_rows = agg.height
curve = curve.with_columns(
    (pl.col("rows_remaining") / total_rows).alias("fraction_remaining")
)
curve_df = curve.to_pandas()

In [ ]:
curve_df.head(10)

In [ ]:
chart = (
    alt.Chart(curve_df)
    .mark_line(point=True)
    .encode(
        x=alt.X("k:Q", title="K (versions_count threshold)"),
        y=alt.Y("rows_remaining:Q", title="Rows remaining"),
        tooltip=[
            alt.Tooltip("k:Q", title="K"),
            alt.Tooltip("rows_remaining:Q", title="Rows remaining"),
            alt.Tooltip("fraction_remaining:Q", title="Fraction remaining", format=".2%")
        ],
    )
    .properties(
        width=700,
        height=400,
        title="Rows remaining after filtering on versions_count >= K"
    )
)

chart

In [ ]:
chart_pct = (
    alt.Chart(curve_df)
    .mark_line(point=True)
    .encode(
        x=alt.X("k:Q", title="K (versions_count threshold)"),
        y=alt.Y("fraction_remaining:Q", title="Fraction remaining", axis=alt.Axis(format="%")),
        tooltip=[
            alt.Tooltip("k:Q", title="K"),
            alt.Tooltip("rows_remaining:Q", title="Rows remaining"),
            alt.Tooltip("fraction_remaining:Q", title="Fraction remaining", format=".2%")
        ],
    )
    .properties(
        width=700,
        height=400,
        title="Fraction of rows remaining after filtering"
    )
)

chart_pct

In [ ]:
K = 100
k_agg = agg.filter(pl.col("versions_count") >= K)
k_agg.shape

## 3. Compute node in/out degrees from aggregated graph

In [ ]:
k_agg.tail(10)

In [ ]:
out_deg = (
    k_agg.group_by("package_name")
    .len()
    .rename({"package_name": "node", "len": "out_degree"})
)

in_deg = (
    k_agg.group_by("dep_name")
    .len()
    .rename({"dep_name": "node", "len": "in_degree"})
)

# Full outer join so we keep nodes that appear only as source or only as target
deg = (
    out_deg.join(in_deg, on="node", how="full", coalesce=True)
    .with_columns([
        pl.col("out_degree").fill_null(0),
        pl.col("in_degree").fill_null(0),
    ])
)

deg_dict = {
    row["node"]: {
        "out_degree": int(row["out_degree"]),
        "in_degree": int(row["in_degree"]),
    }
    for row in deg.iter_rows(named=True)
}

In [ ]:
#deg_dict["pandas"]

## 4. Build PyVis network

In [ ]:
net = Network(
    height="750px",
    width="100%",
    directed=True,
    notebook=True,
    bgcolor="#ffffff",
    font_color="black",
)

# Better default physics for medium graphs
net.barnes_hut()

In [ ]:
sources = set(agg["package_name"].to_list())
targets = set(agg["dep_name"].to_list())
all_nodes = sources | targets

## 5. Add nodes
```   
- label=node name, so package name appears on the node
- size based on in/out degree
```

In [ ]:
for node in all_nodes:
    stats = deg_dict.get(node, {"out_degree": 0, "in_degree": 0})
    out_d = stats["out_degree"]
    in_d = stats["in_degree"]

    # Make popular dependencies a bit larger than high-out-degree packages
    size = 12 + 6 * math.sqrt(2 * in_d + out_d)

    # Optional coloring by role
    if node in sources and node not in targets:
        group = "package"
        color = "#97c2fc"
    elif node in targets and node not in sources:
        group = "dependency"
        color = "#f7a1a1"
    else:
        group = "both"
        color = "#d3d3d3"

    net.add_node(
        node,
        label=node,   # <-- this is what shows the package/dependency name on the node
        size=size,
        color=color,
        title=(
            f"<b>{node}</b><br>"
            f"role: {group}<br>"
            f"in-degree: {in_d}<br>"
            f"out-degree: {out_d}"
        ),
    )

## 6. Add weighted edges

In [ ]:
for row in agg.iter_rows(named=True):
    src = row["package_name"]
    dst = row["dep_name"]
    versions_count = int(row["versions_count"])
    #dep_versions_count = int(row["dep_versions_count"])

    net.add_edge(
        src,
        dst,
        value=versions_count,  # edge thickness
        title=(
            f"<b>{src} → {dst}</b><br>"
            f"package versions using dependency: {versions_count}<br>"
            #f"distinct dependency versions: {dep_versions_count}"
        ),
        arrows="to",
    )


 ## 7. improve label rendering / interaction

In [ ]:
net.set_options("""
var options = {
  "nodes": {
    "shape": "dot",
    "font": {
      "size": 16,
      "face": "arial"
    },
    "scaling": {
      "min": 10,
      "max": 40
    }
  },
  "edges": {
    "smooth": {
      "type": "dynamic"
    },
    "font": {
      "size": 12,
      "align": "middle"
    }
  },
  "physics": {
    "barnesHut": {
      "gravitationalConstant": -3000,
      "springLength": 140,
      "springConstant": 0.04
    },
    "minVelocity": 0.75
  },
  "interaction": {
    "hover": true,
    "navigationButtons": true,
    "keyboard": true
  }
}
""")

## 8. Write Graph to HTML

In [ ]:
output_file = "dependency_graph.html"
net.write_html(output_file, notebook=True)